# E15: Probability Calibration and Score Scaling (Models A, B, C)

**Spec sections**: 9.1 (Platt on DEV OOF only; no isotonic), 9.3 (PDO score mapping, caps), D.7 (`pd_to_score`), 8.2 (calibration metrics), 5.4 (Q1 / Q2 reporting).  
**Inputs**: locked `model_registry.json` (all hashes re-verified), `oof_predictions.parquet`, `calibration/platt_dev_oof.json` (E13), DEV GOOD+BAD (frozen E3 population) and OOT GOOD+BAD via `common.load_window`, `config.scaling`.  
**Outputs**: `calibration/calibration.json`, `reports/06_calibration.xlsx`, `reports/run_summaries/s06_calibrate.json`, MLflow run. The registry, model files and feature log are not modified.

1. Platt `logit(PD) = a + b * logit(p_raw)` for A, B, C fitted on DEV OOF predictions only (never in-sample, never OOT); parameters must equal E13.
2. OOT tests per model, overall / Q1 / Q2, after calibration: CITL, calibration slope, Brier, log loss, decile calibration table (DEV deciles as fixed cut-points).
3. Score mapping with `common.pd_to_score` and `config.scaling`; share of records capped at `score_min` / `score_max`.

The central-tendency (intercept) adjustment of spec 9.2 is applied to the production model after the full-data refit (E16), not here.

In [0]:
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, math, time, pickle, shutil, tempfile, hashlib, importlib, copy, warnings
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
import interpret, optbinning
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="Missing values detected")

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)
SC = cfg["scaling"]
CAL_CFG = cfg["calibration"]

ARTIFACTS = common.artifacts_path(cfg)
REPORTS   = os.path.join(ARTIFACTS, "reports")
RUN_SUMS  = os.path.join(REPORTS, "run_summaries")
CALIB_DIR = os.path.join(ARTIFACTS, "calibration")
REGISTRY_PATH = os.path.join(ARTIFACTS, "model_registry.json")
E13_PLATT_PATH = os.path.join(CALIB_DIR, "platt_dev_oof.json")
CALIB_JSON = os.path.join(CALIB_DIR, "calibration.json")
XLSX_PATH = os.path.join(REPORTS, "06_calibration.xlsx")
RUN_SUMMARY_PATH = os.path.join(RUN_SUMS, "s06_calibrate.json")
for d in (REPORTS, RUN_SUMS, CALIB_DIR):
    os.makedirs(d, exist_ok=True)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = common.table_name(cfg, "master")        # config.tables.master (default acard_master)
IS_VARIANT = common.variant_guard(cfg)               # variant outputs never point at the main model
RUN_CTX    = common.print_run_context(cfg, PROJECT_ROOT)
id_col, date_col, status_col = cfg["columns"]["id"], cfg["columns"]["date"], cfg["columns"]["status"]
LABELLED = cfg["labelled_statuses"]
PSI_STATUSES = cfg["psi_statuses"]
EXP = cfg["expected_counts"]
SPLITS = cfg["oot_report_splits"]                       # Q1 / Q2
PERIODS = ["overall"] + list(SPLITS)
MODELS = ["A", "B", "C"]
CAP_WARN = 0.005                                         # spec 9.3: each capped share normally below 0.5%

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
VERSIONS = {"sklearn": sklearn.__version__, "lightgbm": lgb.__version__, "interpret": interpret.__version__,
            "optbinning": optbinning.__version__, "numpy": np.__version__, "pandas": pd.__version__}

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p) as fh:
        return json.load(fh)

def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)

def canonical(obj):
    return json.loads(json.dumps(obj, sort_keys=True, default=_json_default))

_TMP = tempfile.mkdtemp(prefix="e15_")
def to_volume(dst, writer):
    """Write to a local temp file, then copy to the UC volume (FUSE does not allow chmod/utime)."""
    tmp = os.path.join(_TMP, os.path.basename(dst))
    writer(tmp)
    shutil.copyfile(tmp, dst)

_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]}")
print(f"Calibration : {CAL_CFG} | Scaling: {SC}")

In [0]:
REG = read_json(REGISTRY_PATH)
assert REG.get("locked") is True, "model_registry.json is not locked: run E12 (s04_lock) first"
for k in ("decision_rules_signed_by", "signed_date"):
    assert str(REG.get("signoff", {}).get(k, "")).strip(), f"registry sign-off field {k} is empty"
HASH_CHECK = {k: sha256_file(v["path"]) == v["sha256"] for k, v in REG["files"].items()}
bad = [k for k, ok in HASH_CHECK.items() if not ok]
assert not bad, f"ABORT: {len(bad)} locked file(s) changed since the lock: {bad}"
REG_SNAPSHOT_SHA = hashlib.sha256(json.dumps(canonical(REG), sort_keys=True).encode()).hexdigest()
FILES = {k: v["path"] for k, v in REG["files"].items()}

# Calibration method is pre-registered (spec 9.1): Platt on DEV OOF; isotonic prohibited
PRE_CAL = REG["pre_registered"]["calibration"]
assert PRE_CAL["method"] == "platt" and PRE_CAL["fit_on"] == "dev_oof", f"registry calibration is {PRE_CAL}"
assert CAL_CFG["method"] == "platt" and CAL_CFG["fit_on"] == "dev_oof", f"config calibration is {CAL_CFG}"
assert SC["odds_type"] == "good_to_bad", "spec 9.3: scores are defined on Good:Bad odds"

A_FEATS = REG["features"]["A"]
POOL    = REG["features"]["BC_pool"]
BC_COLS = REG["features"]["BC_columns"]
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
SPEC = read_json(FILES["models/model_spec.json"])
WOE_KW = SPEC["A"]["woe_transform"]
A_BP = pickle.load(open(FILES["models/A/binning_process.pkl"], "rb"))
A_LR = pickle.load(open(FILES["models/A/lr_sklearn.pkl"], "rb"))
B_EBM = pickle.load(open(FILES["models/B/ebm.pkl"], "rb"))
C_BST = lgb.Booster(model_file=FILES["models/C/lgbm.txt"])
assert C_BST.num_trees() == REG["lightgbm_rounds"]["final_rounds"], "LightGBM trees != locked final rounds"

def score_raw(df):
    """Raw (uncalibrated) Bad probability of the locked A, B, C (same as E13). Higher = riskier."""
    XA = df[A_FEATS].apply(pd.to_numeric, errors="coerce")
    pa = A_LR.predict_proba(np.asarray(A_BP.transform(XA, metric="woe", **WOE_KW), dtype=float))[:, 1]
    XR = common.prepare_raw(df, POOL, fdict)
    assert list(XR.columns) == BC_COLS, "prepare_raw columns differ from the locked B/C columns"
    return {"A": pa, "B": B_EBM.predict_proba(XR)[:, 1], "C": C_BST.predict(XR)}

print(f"Registry v{REG['version']} locked at {REG['locked_at']}; {len(HASH_CHECK)} file hashes re-verified")
print(f"Pre-registered calibration: {PRE_CAL['method']} on {PRE_CAL['fit_on']} ({PRE_CAL['formula']})")

In [0]:
# Model lock is in place, so OOT may be read (only through common.load_window). OOT is used for TESTS only.
master = spark.table(MASTER_TBL).filter("window IN ('DEV', 'OOT')").toPandas()
dev_all = common.load_window(master, "DEV", cfg)
oot_all = common.load_window(master, "OOT", cfg)
del master

frozen_ids = set(pd.read_parquet(FILES["folds/r_folds.parquet"], columns=["app_id"])["app_id"].astype(str))
dev_gb = dev_all[dev_all[status_col].isin(LABELLED) & dev_all[id_col].astype(str).isin(frozen_ids)].reset_index(drop=True)
oot_gb = oot_all[oot_all[status_col].isin(LABELLED)].reset_index(drop=True)
common.assert_labelled(dev_gb, cfg); common.assert_labelled(oot_gb, cfg)
dev_p1 = dev_all[dev_all[status_col].isin(PSI_STATUSES)].reset_index(drop=True)     # all scored applicants (caps only)
oot_p1 = oot_all[oot_all[status_col].isin(PSI_STATUSES)].reset_index(drop=True)
del dev_all, oot_all

y_dev = (dev_gb[status_col] == "BAD").astype(int).to_numpy()
y_oot = (oot_gb[status_col] == "BAD").astype(int).to_numpy()
assert len(dev_gb) == EXP["DEV"]["records"] and int(y_dev.sum()) == EXP["DEV"]["bads"], "DEV counts differ from expected_counts"
assert len(oot_gb) == EXP["OOT"]["records"] and int(y_oot.sum()) == EXP["OOT"]["bads"], "OOT counts differ from expected_counts"
DEV_IDS = dev_gb[id_col].astype(str)
OOT_IDS = set(oot_gb[id_col].astype(str))
assert DEV_IDS.is_unique and not (set(DEV_IDS) & OOT_IDS), "DEV ids not unique or overlap OOT"

def pmask(df, p):
    if p == "overall":
        return np.ones(len(df), bool)
    s, e = SPLITS[p]
    d = pd.to_datetime(df[date_col])
    return ((d >= s) & (d < e)).to_numpy()

COUNTS = pd.DataFrame([{"sample": "DEV GOOD+BAD", "period": "all", "records": len(dev_gb), "bads": int(y_dev.sum())}]
                      + [{"sample": "OOT GOOD+BAD", "period": p, "records": int(pmask(oot_gb, p).sum()),
                          "bads": int(y_oot[pmask(oot_gb, p)].sum())} for p in PERIODS]
                      + [{"sample": "DEV P1 (scored, caps only)", "period": "all", "records": len(dev_p1), "bads": None},
                         {"sample": "OOT P1 (scored, caps only)", "period": "overall", "records": len(oot_p1), "bads": None}])
COUNTS["bad_rate"] = COUNTS["bads"] / COUNTS["records"]
display(COUNTS)

In [0]:
# ── Spec 9.1: fit on the DEV OOF predictions (oof_predictions.parquet, hash-locked) and actual labels ONLY ──
oof = pd.read_parquet(FILES["oof_predictions.parquet"])
oof["app_id"] = oof["app_id"].astype(str)
assert list(oof.columns) == ["app_id", "model", "fold", "p_oof"], f"unexpected OOF columns {list(oof.columns)}"
ylab = pd.Series(y_dev, index=DEV_IDS)

FIT_LOG, PLATT, OOF_P = [], {}, {}
for m in MODELS:
    g = oof[oof["model"] == m].set_index("app_id")
    assert g.index.is_unique and set(g.index) == set(DEV_IDS), f"OOF for {m} does not cover DEV GOOD+BAD exactly once"
    yy, pp = ylab.loc[g.index].to_numpy(), g["p_oof"].to_numpy()
    FIT_LOG.append({"model": m, "source": "oof_predictions.parquet", "n": len(yy), "n_bad": int(yy.sum()), "ids": set(g.index)})
    PLATT[m] = common.fit_platt(yy, pp)
    OOF_P[m] = (g.index.to_numpy(), yy, pp)

# Evidence the inputs are OOF, not in-sample: the locked full-DEV models' in-sample PDs differ from p_oof
RAW_DEV = score_raw(dev_gb)
raw_dev_by_id = {m: pd.Series(RAW_DEV[m], index=DEV_IDS) for m in MODELS}
INSAMPLE_VS_OOF = pd.DataFrame([{"model": m,
    "mean_abs_diff_oof_vs_insample": float(np.abs(OOF_P[m][2] - raw_dev_by_id[m].loc[OOF_P[m][0]].to_numpy()).mean()),
    "auc_oof": roc_auc_score(OOF_P[m][1], OOF_P[m][2]),
    "auc_insample": roc_auc_score(OOF_P[m][1], raw_dev_by_id[m].loc[OOF_P[m][0]].to_numpy())} for m in MODELS])

# ── E13 parameters (calibration/platt_dev_oof.json) ──
E13 = read_json(E13_PLATT_PATH)
E13_SHA = sha256_file(E13_PLATT_PATH)
assert E13["method"] == "platt" and E13["fit_on"] == "dev_oof", "E13 Platt file is not Platt on DEV OOF"
assert E13["oof_sha256"] == REG["files"]["oof_predictions.parquet"]["sha256"], "E13 used a different OOF file"
PLATT_TAB = pd.DataFrame([{"model": m, "a": PLATT[m]["a"], "b": PLATT[m]["b"], "n": PLATT[m]["n"], "n_bad": PLATT[m]["n_bad"],
                           "a_e13": E13["params"][m]["a"], "b_e13": E13["params"][m]["b"],
                           "n_e13": E13["params"][m]["n"], "n_bad_e13": E13["params"][m]["n_bad"]} for m in MODELS])
PLATT_TAB["abs_diff_a"] = (PLATT_TAB["a"] - PLATT_TAB["a_e13"]).abs()
PLATT_TAB["abs_diff_b"] = (PLATT_TAB["b"] - PLATT_TAB["b_e13"]).abs()
PLATT_TOL = 1e-9
PLATT_EQUAL_E13 = bool((PLATT_TAB[["abs_diff_a", "abs_diff_b"]].max().max() <= PLATT_TOL)
                       and (PLATT_TAB["n"] == PLATT_TAB["n_e13"]).all() and (PLATT_TAB["n_bad"] == PLATT_TAB["n_bad_e13"]).all())

# MLE property: calibrated mean PD on the fitting data equals the DEV bad rate
OOF_MEAN_GAP = {m: float(common.apply_platt(OOF_P[m][2], PLATT[m]).mean() - OOF_P[m][1].mean()) for m in MODELS}
print(f"Platt equals E13 (tol {PLATT_TOL}): {PLATT_EQUAL_E13} | mean PD - DEV bad rate on OOF: "
      f"{ {m: f'{v:.2e}' for m, v in OOF_MEAN_GAP.items()} }")
print(PLATT_TAB.round(8).to_string(index=False))
print(INSAMPLE_VS_OOF.round(4).to_string(index=False))

In [0]:
RAW = {"dev_gb": RAW_DEV, "oot_gb": score_raw(oot_gb), "dev_p1": score_raw(dev_p1), "oot_p1": score_raw(oot_p1)}
CAL = {k: {m: common.apply_platt(v[m], PLATT[m]) for m in MODELS} for k, v in RAW.items()}
for k, v in CAL.items():
    for m in MODELS:
        assert np.isfinite(v[m]).all() and ((v[m] > 0) & (v[m] < 1)).all(), f"{k} {m}: calibrated PD outside (0, 1)"

# ── CITL, calibration slope, Brier, log loss (spec 8.2 / 9.1), after calibration; DEV OOF shown for reference ──
cal_rows = []
for m in MODELS:
    _, yy, pp = OOF_P[m]
    cal_rows.append({"model": m, "sample": "DEV OOF (fit data)", "period": "all", "n": len(yy), "n_bad": int(yy.sum()),
                     **common.calibration_stats(yy, common.apply_platt(pp, PLATT[m])),
                     "log_loss_raw": common.calibration_stats(yy, pp)["log_loss"]})
    for p in PERIODS:
        mk = pmask(oot_gb, p)
        yy, pr, pc = y_oot[mk], RAW["oot_gb"][m][mk], CAL["oot_gb"][m][mk]
        cal_rows.append({"model": m, "sample": "OOT", "period": p, "n": int(mk.sum()), "n_bad": int(yy.sum()),
                         **common.calibration_stats(yy, pc), "log_loss_raw": common.calibration_stats(yy, pr)["log_loss"]})
CALIB = pd.DataFrame(cal_rows)
CALIB["ratio_mean_pd_to_bad_rate"] = CALIB["mean_pd"] / CALIB["bad_rate"]

# ── Decile calibration table: DEV deciles of each model's calibrated PD (locked model) as fixed cut-points ──
def edges(p, q):
    return np.unique(np.quantile(p, np.linspace(0, 1, q + 1)[1:-1]))
def band(p, e):
    return np.searchsorted(e, p, side="right") + 1          # 1 = lowest PD ... 10 = highest PD
EDGES = {m: edges(CAL["dev_gb"][m], 10) for m in MODELS}
dec_parts = []
for m in MODELS:
    for p in PERIODS:
        mk = pmask(oot_gb, p)
        t = pd.DataFrame({"decile": band(CAL["oot_gb"][m][mk], EDGES[m]), "y": y_oot[mk], "pd": CAL["oot_gb"][m][mk]})
        g = (t.groupby("decile").agg(n=("y", "size"), bads=("y", "sum"), mean_pd=("pd", "mean"))
              .reindex(range(1, len(EDGES[m]) + 2)).reset_index())
        g["bad_rate"] = g["bads"] / g["n"]
        g["pd_minus_bad_rate"] = g["mean_pd"] - g["bad_rate"]
        g["expected_bads"] = g["mean_pd"] * g["n"]
        g.insert(0, "period", p); g.insert(0, "model", m)
        dec_parts.append(g)
DECILES = pd.concat(dec_parts, ignore_index=True)

# Cross-check (information only): E13 calibration sheet in reports/05_oot.xlsx, same models / Platt / OOT
E13_XCHECK = pd.DataFrame()
try:
    x13 = os.path.join(REPORTS, "05_oot.xlsx")
    for sh in pd.ExcelFile(x13).sheet_names:
        t = pd.read_excel(x13, sheet_name=sh)
        if {"model", "period", "log_loss", "citl"} <= set(t.columns):
            mine = CALIB[CALIB["sample"] == "OOT"].set_index(["model", "period"])
            t = t.set_index(["model", "period"])
            E13_XCHECK = pd.DataFrame([{"model": m, "period": p, "sheet": sh,
                                        "log_loss_e15": mine.loc[(m, p), "log_loss"], "log_loss_e13": t.loc[(m, p), "log_loss"],
                                        "citl_e15": mine.loc[(m, p), "citl"], "citl_e13": t.loc[(m, p), "citl"]}
                                       for m in MODELS for p in PERIODS])
            E13_XCHECK["max_abs_diff"] = np.maximum((E13_XCHECK.log_loss_e15 - E13_XCHECK.log_loss_e13).abs(),
                                                    (E13_XCHECK.citl_e15 - E13_XCHECK.citl_e13).abs())
            break
    print(f"E13 cross-check: max abs diff {E13_XCHECK['max_abs_diff'].max():.2e}" if len(E13_XCHECK) else "E13 calibration sheet not found")
except Exception as ex:
    print(f"E13 cross-check not available: {type(ex).__name__}: {ex}")

print(CALIB[["model", "sample", "period", "n", "n_bad", "bad_rate", "mean_pd", "citl", "cal_intercept", "cal_slope", "brier",
             "log_loss", "log_loss_raw"]].round(4).to_string(index=False))
display(DECILES[DECILES.period == "overall"].round(4))

In [0]:
FACTOR = SC["pdo"] / np.log(2)
OFFSET = SC["base_score"] - FACTOR * np.log(SC["base_odds"])
LO, HI = SC["score_min"], SC["score_max"]

def score_unclipped(pd_):
    """Spec 9.3 formula before rounding / capping (PD clipped as in common.pd_to_score)."""
    p = np.clip(np.asarray(pd_, dtype=float), 1e-6, 1 - 1e-6)
    return OFFSET + FACTOR * np.log((1 - p) / p)

SCORE = {k: {m: common.pd_to_score(v[m], cfg) for m in MODELS} for k, v in CAL.items()}
POPS = {"dev_gb": ("DEV GOOD+BAD", dev_gb), "oot_gb": ("OOT GOOD+BAD", oot_gb),
        "dev_p1": ("DEV P1 (all scored)", dev_p1), "oot_p1": ("OOT P1 (all scored)", oot_p1)}
cap_rows, dist_rows = [], []
for k, (label, df) in POPS.items():
    periods = PERIODS if k.startswith("oot") else ["all"]
    for m in MODELS:
        for p in periods:
            mk = pmask(df, p) if p != "all" else np.ones(len(df), bool)
            s, r = SCORE[k][m][mk], np.round(score_unclipped(CAL[k][m][mk]))
            assert np.array_equal(s, np.clip(r, LO, HI).astype(int)), f"pd_to_score differs from the 9.3 formula ({k} {m})"
            cap_rows.append({"population": label, "model": m, "period": p, "n": int(mk.sum()),
                             "capped_at_min_share": float((r < LO).mean()), "capped_at_max_share": float((r > HI).mean()),
                             "n_capped_min": int((r < LO).sum()), "n_capped_max": int((r > HI).sum()),
                             "at_min_share": float((s == LO).mean()), "at_max_share": float((s == HI).mean())})
            q = np.percentile(s, [0, 1, 5, 25, 50, 75, 95, 99, 100])
            dist_rows.append({"population": label, "model": m, "period": p, "n": int(mk.sum()), "mean": float(s.mean()),
                              **{f"p{int(x)}": v for x, v in zip([0, 1, 5, 25, 50, 75, 95, 99, 100], q)}})
CAPS = pd.DataFrame(cap_rows)
CAPS["warn_gt_0_5pct"] = (CAPS["capped_at_min_share"] > CAP_WARN) | (CAPS["capped_at_max_share"] > CAP_WARN)
SCORE_DIST = pd.DataFrame(dist_rows)

# Mapping examples (anchors of spec 9.3) and the score at each Platt-calibrated DEV decile edge
EXAMPLES = pd.DataFrame({"pd": [1 / 11, 1 / 21, 1 / 41, 0.5, 1e-9, 1 - 1e-9]})
EXAMPLES["good_bad_odds"] = (1 - EXAMPLES["pd"]) / EXAMPLES["pd"]
EXAMPLES["score_unclipped"] = score_unclipped(EXAMPLES["pd"])
EXAMPLES["score"] = common.pd_to_score(EXAMPLES["pd"].to_numpy(), cfg)

print(f"Factor = {FACTOR:.6f} | Offset = {OFFSET:.6f} | caps [{LO}, {HI}]")
print(CAPS[CAPS.period.isin(["all", "overall"])][["population", "model", "n", "capped_at_min_share", "capped_at_max_share",
                                                  "warn_gt_0_5pct"]].to_string(index=False))
for r in CAPS[CAPS.warn_gt_0_5pct].itertuples():
    print(f"WARNING: {r.population} {r.model} {r.period}: capped share min {r.capped_at_min_share:.3%} / max {r.capped_at_max_share:.3%} > 0.5%")
print(EXAMPLES.round(4).to_string(index=False))

In [0]:
RUN_TIME = datetime.now().isoformat(timespec="seconds")
CALIB_DOC = {
    "module": "E15_calibrate", "run_time": RUN_TIME, "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
    "registry_version": REG["version"],
    "method": "platt", "formula": "logit(PD) = a + b * logit(p_raw)", "fit_on": "dev_oof",
    "isotonic": "prohibited (spec 9.1); not used",
    "oof_file": FILES["oof_predictions.parquet"], "oof_sha256": REG["files"]["oof_predictions.parquet"]["sha256"],
    "e13_platt_file": E13_PLATT_PATH, "e13_platt_sha256": E13_SHA, "equal_to_e13": PLATT_EQUAL_E13,
    "logit_clip": 1e-6,
    "models": {m: {"a": PLATT[m]["a"], "b": PLATT[m]["b"], "n": PLATT[m]["n"], "n_bad": PLATT[m]["n_bad"]} for m in MODELS},
    "scaling": {k: SC[k] for k in ("base_score", "base_odds", "odds_type", "pdo", "score_min", "score_max")},
    "factor": float(FACTOR), "offset": float(OFFSET),
    "score_formula": "score = clip(round(offset + factor * ln((1 - PD) / PD)), score_min, score_max); PD clipped to [1e-6, 1 - 1e-6]",
    "central_tendency": f"not applied here; spec 9.2 intercept adjustment on the production model after E16 (config {CAL_CFG.get('central_tendency')})",
}
to_volume(CALIB_JSON, lambda p: json.dump(CALIB_DOC, open(p, "w"), indent=2, default=_json_default))
CALIB_JSON_SHA = sha256_file(CALIB_JSON)
print(f"Saved {CALIB_JSON} (sha256 {CALIB_JSON_SHA[:12]})")
print(json.dumps({k: CALIB_DOC[k] for k in ("models", "scaling", "factor", "offset")}, indent=1))

In [0]:
META = pd.DataFrame([{"module": "E15_calibrate", "run_time": RUN_TIME, "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
                      "registry_version": REG["version"], "calibration_json_sha256": CALIB_JSON_SHA, **VERSIONS}])
NOTES = pd.DataFrame({"note": [
    "Platt (logistic) calibration logit(PD) = a + b * logit(p_raw), fitted on DEV OOF predictions (R-fold repeat 1, spec 7.6) "
    "and actual labels only. Not fitted on in-sample predictions or on OOT. Isotonic regression is not used (spec 9.1).",
    f"Parameters equal calibration/platt_dev_oof.json from E13 (tolerance {PLATT_TOL}): {PLATT_EQUAL_E13}.",
    "OOT tests after calibration, overall / 2025Q1 / 2025Q2 (spec 5.4): CITL = mean PD - Bad Rate; calibration slope = "
    "coefficient of a logistic regression of the outcome on logit(PD); Brier; log loss (raw log loss shown for comparison).",
    "Decile calibration table: DEV deciles of each model's calibrated PD (locked full-DEV model) as fixed cut-points, applied to OOT.",
    f"Score mapping (spec 9.3, D.7): Factor = PDO / ln 2 = {FACTOR:.4f}, Offset = {OFFSET:.4f}; higher score = lower risk; "
    f"capped to [{LO}, {HI}]. 'capped' = rounded formula score outside the range before the cap.",
    "PD = P(Bad | Good or Bad, approved); Indeterminate excluded (spec 9.2). The 9.2 central-tendency adjustment is not applied here.",
]})

def _write_xlsx(p):
    with pd.ExcelWriter(p, engine="openpyxl") as w:
        META.to_excel(w, sheet_name="meta", index=False)
        NOTES.to_excel(w, sheet_name="note", index=False)
        COUNTS.to_excel(w, sheet_name="counts", index=False)
        PLATT_TAB.to_excel(w, sheet_name="platt_params", index=False)
        INSAMPLE_VS_OOF.to_excel(w, sheet_name="fit_input_check", index=False)
        CALIB.to_excel(w, sheet_name="calibration_stats", index=False)
        DECILES.to_excel(w, sheet_name="decile_calibration", index=False)
        CAPS.to_excel(w, sheet_name="score_caps", index=False)
        SCORE_DIST.to_excel(w, sheet_name="score_distribution", index=False)
        EXAMPLES.to_excel(w, sheet_name="score_mapping_examples", index=False)
        pd.DataFrame([{"factor": FACTOR, "offset": OFFSET, **CALIB_DOC["scaling"]}]).to_excel(w, sheet_name="scaling", index=False)
        if len(E13_XCHECK):
            E13_XCHECK.to_excel(w, sheet_name="e13_crosscheck", index=False)
to_volume(XLSX_PATH, _write_xlsx)
print(f"Saved {XLSX_PATH} ({len(pd.ExcelFile(XLSX_PATH).sheet_names)} sheets)")

In [0]:
import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
co = CALIB[CALIB["sample"] == "OOT"].set_index(["model", "period"])
cap_main = CAPS[CAPS.period.isin(["all", "overall"])]
with mlflow.start_run(run_name="E15_calibrate") as run:
    mlflow.set_tags({"module": "E15_calibrate", "git_commit": GIT_COMMIT, "registry_version": REG["version"],
                     "method": "platt_dev_oof"})
    mlflow.log_params({"config_hash": CONFIG_HASH, "seed": SEED, "factor": round(FACTOR, 6), "offset": round(OFFSET, 6),
                       **{f"{m}_platt_{k}": round(PLATT[m][k], 8) for m in MODELS for k in ("a", "b")},
                       **{f"scaling_{k}": v for k, v in CALIB_DOC["scaling"].items()}})
    mlflow.log_metrics({**{f"{m}_oot_{p}_{k}": float(co.loc[(m, p), k]) for m in MODELS for p in PERIODS
                           for k in ("citl", "cal_slope", "brier", "log_loss")},
                        **{f"{m}_{r.population.split()[0].lower()}_{r.population.split()[1].lower().strip('()')}_cap_{e}":
                           float(getattr(r, f"capped_at_{e}_share")) for r in cap_main.itertuples() for m in [r.model] for e in ("min", "max")}})
    for p in (XLSX_PATH, CALIB_JSON):
        mlflow.log_artifact(p)
    MLFLOW_RUN_ID = run.info.run_id
print(f"MLflow run: {MLFLOW_RUN_ID}")

In [0]:
# ── Acceptance (E15): re-read outputs and check every criterion ──
print("=" * 60); print("ACCEPTANCE CRITERIA (E15 — calibration and score scaling)"); print("=" * 60)
REG2 = read_json(REGISTRY_PATH)
reg_unchanged = hashlib.sha256(json.dumps(canonical(REG2), sort_keys=True).encode()).hexdigest() == REG_SNAPSHOT_SHA
CJ = read_json(CALIB_JSON)
xl = pd.ExcelFile(XLSX_PATH)
meta_r = pd.read_excel(XLSX_PATH, sheet_name="meta")
caps_r = pd.read_excel(XLSX_PATH, sheet_name="score_caps")
cal_r = pd.read_excel(XLSX_PATH, sheet_name="calibration_stats")
dec_r = pd.read_excel(XLSX_PATH, sheet_name="decile_calibration")

# Fit inputs: exactly the DEV GOOD+BAD ids from the OOF file, never OOT, three fits only
fit_oof_only = (len(FIT_LOG) == len(MODELS) and all(e["source"] == "oof_predictions.parquet" and e["ids"] == set(DEV_IDS)
                                                    and not (e["ids"] & OOT_IDS) for e in FIT_LOG))
not_insample = bool((INSAMPLE_VS_OOF["mean_abs_diff_oof_vs_insample"] > 1e-6).all())
no_isotonic = ("IsotonicRegression" not in globals() and CJ["method"] == "platt" and CJ["fit_on"] == "dev_oof")
# Independent refit of the Platt parameters with statsmodels (unpenalised MLE) on the same OOF data
import statsmodels.api as sm
sm_diff = max(max(abs(sm.Logit(OOF_P[m][1], sm.add_constant(common.logit(OOF_P[m][2]))).fit(disp=0).params[i] - PLATT[m][k])
                  for i, k in ((0, "a"), (1, "b"))) for m in MODELS)

# ── Unit tests: score mapping (spec 9.3 / 16.3) ──
UT = [("PD = 1/21 -> 600", int(common.pd_to_score([1 / 21], cfg)[0]), 600),
      ("PD = 1/41 -> 620", int(common.pd_to_score([1 / 41], cfg)[0]), 620),
      ("PD = 1/11 -> 580", int(common.pd_to_score([1 / 11], cfg)[0]), 580),
      (f"PD -> 0 capped at score_max {HI}", int(common.pd_to_score([1e-12], cfg)[0]), HI),
      (f"PD -> 1 capped at score_min {LO}", int(common.pd_to_score([1 - 1e-12], cfg)[0]), LO)]
grid = np.linspace(0.001, 0.999, 999)
mono_ok = bool((np.diff(common.pd_to_score(grid, cfg)) <= 0).all())
factor_ok = abs(FACTOR - 28.8539) < 1e-4 and abs(OFFSET - 513.56) < 0.01 if (SC["pdo"], SC["base_score"], SC["base_odds"]) == (20, 600, 20) else True
print("Unit tests (score mapping):")
for name, got, exp in UT:
    print(f"  {'✅' if got == exp else '❌'} {name}: got {got}")

# Capped shares: reported; WARN (not fail) above 0.5%
cap_main = caps_r[caps_r["period"].isin(["all", "overall"])]
CAP_WARNINGS = [f"{r.population} {r.model} {r.period}: min {r.capped_at_min_share:.3%}, max {r.capped_at_max_share:.3%}"
                for r in caps_r.itertuples() if r.capped_at_min_share > CAP_WARN or r.capped_at_max_share > CAP_WARN]

checks = [
    ("Registry locked, all locked file hashes match, registry content unchanged by E15",
     REG2.get("locked") is True and all(sha256_file(v["path"]) == v["sha256"] for v in REG2["files"].values()) and reg_unchanged),
    ("Platt fitted on DEV OOF only (3 fits; inputs = the DEV GOOD+BAD ids of oof_predictions.parquet; no OOT id)", fit_oof_only),
    ("Fit inputs are OOF, not in-sample (p_oof differs from the locked full-DEV model's PD for every model)", not_insample),
    ("Method = Platt; no isotonic regression", no_isotonic),
    (f"Platt a, b, n, n_bad equal calibration/platt_dev_oof.json from E13 (max |diff| "
     f"{PLATT_TAB[['abs_diff_a', 'abs_diff_b']].max().max():.1e})", PLATT_EQUAL_E13),
    (f"Independent statsmodels refit agrees (max |diff| {sm_diff:.1e})", sm_diff < 1e-5),
    ("Calibrated mean PD on the fitting data equals the DEV bad rate (MLE)", max(abs(v) for v in OOF_MEAN_GAP.values()) < 1e-4),
    ("OOT CITL, slope, Brier, log loss for A, B, C x overall / Q1 / Q2 in the report",
     len(cal_r[cal_r["sample"] == "OOT"]) == len(MODELS) * len(PERIODS)
     and cal_r.loc[cal_r["sample"] == "OOT", ["citl", "cal_slope", "brier", "log_loss"]].notna().all().all()),
    ("Decile calibration table for A, B, C x overall / Q1 / Q2; counts add up to OOT",
     all(int(dec_r[(dec_r.model == m) & (dec_r.period == p)]["n"].sum()) == int(pmask(oot_gb, p).sum())
         for m in MODELS for p in PERIODS)),
    ("Score unit tests: 1/21 -> 600, 1/41 -> 620, 1/11 -> 580, caps at both ends", all(got == exp for _, got, exp in UT)),
    (f"Score monotonic decreasing in PD; Factor {FACTOR:.4f}, Offset {OFFSET:.2f} as spec 9.3", mono_ok and factor_ok),
    ("Capped shares at score_min / score_max reported for DEV / OOT (GOOD+BAD and all scored) x A, B, C",
     len(cap_main) == 4 * len(MODELS) and cap_main[["capped_at_min_share", "capped_at_max_share"]].notna().all().all()),
    ("calibration.json has a, b per model, scaling parameters, Factor, Offset (re-read)",
     set(CJ["models"]) == set(MODELS) and all(CJ["models"][m]["a"] == PLATT[m]["a"] and CJ["models"][m]["b"] == PLATT[m]["b"]
                                              for m in MODELS)
     and CJ["scaling"] == {k: SC[k] for k in CJ["scaling"]} and set(CJ["scaling"]) >= {"base_score", "base_odds", "pdo", "score_min", "score_max"}
     and abs(CJ["factor"] - FACTOR) < 1e-12 and abs(CJ["offset"] - OFFSET) < 1e-12),
    ("Report: first sheet 'meta' (module, run time, git commit, config sha256)",
     xl.sheet_names[0] == "meta" and {"module", "run_time", "git_commit", "config_sha256"} <= set(meta_r.columns)),
]
print()
for name, ok in checks:
    print(f"  {'✅ PASS' if ok else '❌ FAIL'} : {name}")
if CAP_WARNINGS:
    warnings.warn("Capped share above 0.5%: " + "; ".join(CAP_WARNINGS))
    print("\n⚠️  WARNING capped share > 0.5%:\n   " + "\n   ".join(CAP_WARNINGS))
else:
    print("\nCapped shares: all <= 0.5% (no warning)")
ALL_PASS = all(ok for _, ok in checks)

co = CALIB[CALIB["sample"] == "OOT"].set_index(["model", "period"])
RUN_SUMMARY = {
    "module": "E15_calibrate", "run_time": RUN_TIME, "duration_s": round(time.time() - _t0, 1), "git_commit": GIT_COMMIT,
    "config_sha256": CONFIG_HASH, "registry_version": REG2["version"], "mlflow_run_id": MLFLOW_RUN_ID,
    "row_counts": {"dev_gb": len(dev_gb), "dev_bads": int(y_dev.sum()), "oot_gb": len(oot_gb), "oot_bads": int(y_oot.sum()),
                   "dev_p1": len(dev_p1), "oot_p1": len(oot_p1)},
    "platt": {m: {"a": PLATT[m]["a"], "b": PLATT[m]["b"]} for m in MODELS}, "factor": FACTOR, "offset": OFFSET,
    "oot_calibration": {f"{m}_{p}": {k: round(float(co.loc[(m, p), k]), 6) for k in ("citl", "cal_slope", "brier", "log_loss")}
                        for m in MODELS for p in PERIODS},
    "capped_shares": {f"{r.population}|{r.model}": {"min": r.capped_at_min_share, "max": r.capped_at_max_share}
                      for r in cap_main.itertuples()},
    "cap_warnings": CAP_WARNINGS,
    "unit_tests": {name: got == exp for name, got, exp in UT},
    "acceptance": {name: bool(ok) for name, ok in checks}, "all_pass": ALL_PASS,
}
to_volume(RUN_SUMMARY_PATH, lambda p: json.dump(RUN_SUMMARY, open(p, "w"), indent=2, default=_json_default))
print(f"\nRun summary: {RUN_SUMMARY_PATH}")

for name, got, exp in UT:
    assert got == exp, f"Score unit test failed: {name} (got {got})"
for name, ok in checks:
    assert ok, f"E15 acceptance failed: {name}"
print(f"\nE15 acceptance: {len(checks)}/{len(checks)} PASS | unit tests {len(UT)}/{len(UT)} PASS | cap warnings: {len(CAP_WARNINGS)}")